# Data Reweighting by Kamiran and Calders - Adult data

This notebook contains an implementation of the **pre-processing** fairness intervention introduced in [Data preprocessing techniques for classification without discrimination](https://link.springer.com/article/10.1007/s10115-011-0463-8) by Kamiran and Calders (2012), as implemented in the IBM AIF360 fairness toolbox (github.com/Trusted-AI/AIF360).

Unlike Fairness Through Unawareness (a weak pre-processing baseline) or Hardt et al.'s decision-threshold correction (a post-processing intervention), reweighing acts on the *training data itself*, before a model is ever fit.

The intervention works by assigning each training example a weight based on how over- or under-represented its (protected attribute, label) combination is, relative to what we would expect if the protected attribute and the label were statistically independent. Concretely, for a protected attribute value $s$ and label $y$:

$$w(s, y) = \frac{P(S = s) \cdot P(Y = y)}{P(S = s, Y = y)}$$

Combinations that are under-represented (e.g. women with high income, in the Adult data) are up-weighted; combinations that are over-represented are down-weighted. A model trained with these weights is trained on a *reweighted* — not resampled — version of the data, so no rows are duplicated or discarded.

We evaluate the effect of reweighing on **demographic parity**, the fairness notion it directly targets.

In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from aif360.algorithms.preprocessing import Reweighing
from aif360.datasets import StandardDataset
from fairlearn.metrics import demographic_parity_difference, demographic_parity_ratio
from sklearn.linear_model import LogisticRegression

from helpers.metrics import accuracy
from helpers.plot import group_box_plots

pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[AdversarialDebiasing]'


pip install 'aif360[inFairness]'


The baseline/FTU models were pickled with an older scikit-learn version, and this notebook pulls in libraries with newer, stricter validation than the versions the original CDEI project pinned in 2020. The cell below silences the resulting (harmless) version-compatibility warnings so the output stays focused on the fairness results.

In [2]:
import warnings

warnings.filterwarnings("ignore")

## Load data

We use the same preprocessed Adult census data as the FTU and Hardt et al. notebooks. Check out `artifacts/README.md` for details on how this data was obtained.

In [3]:
artifacts_dir = Path("../../../artifacts")

In [4]:
data_dir = artifacts_dir / "data" / "adult"

train = pd.read_csv(data_dir / "processed" / "train-one-hot.csv")
val = pd.read_csv(data_dir / "processed" / "val-one-hot.csv")
test = pd.read_csv(data_dir / "processed" / "test-one-hot.csv")

As with the other intervention notebooks, we wrap the data in AIF360's `StandardDataset` objects, which record which column is the protected attribute and which is the label.

In [5]:
train_sds = StandardDataset(
    train,
    label_name="salary",
    favorable_classes=[1],
    protected_attribute_names=["sex"],
    privileged_classes=[[1]],
)
val_sds = StandardDataset(
    val,
    label_name="salary",
    favorable_classes=[1],
    protected_attribute_names=["sex"],
    privileged_classes=[[1]],
)
test_sds = StandardDataset(
    test,
    label_name="salary",
    favorable_classes=[1],
    protected_attribute_names=["sex"],
    privileged_classes=[[1]],
)

Define which binary value goes with the (un-)privileged group.

In [6]:
privileged_groups = [{"sex": 1.0}]
unprivileged_groups = [{"sex": 0.0}]

## Load baseline model

For consistency with the other notebooks we load the same baseline model, trained on the original, unmodified data with no fairness intervention applied.

In [7]:
baseline_model = joblib.load(artifacts_dir / "models" / "finance" / "baseline.pkl")

bl_test_probs = baseline_model.predict_proba(test.drop("salary", axis=1))[:, 1]
bl_test_pred = (bl_test_probs > 0.5).astype(int)

## Reweigh the training data

We learn the reweighing transformation on the training data, then apply it to the validation data. Because reweighing changes *sample weights* rather than the data itself, we train a fresh logistic regression model on the validation set, passing the learnt weights via `sample_weight`.

In [8]:
RW = Reweighing(
    unprivileged_groups=unprivileged_groups,
    privileged_groups=privileged_groups,
)
RW.fit(train_sds)

Apply the transformation to the validation data, then train a fair model using the resulting instance weights.

In [9]:
val_sds_transf = RW.transform(val_sds)

model_fair = LogisticRegression(max_iter=10000)
X_val = val_sds_transf.features
y_val = val_sds_transf.labels.flatten()
model_fair.fit(X_val, y_val, sample_weight=val_sds_transf.instance_weights)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

Apply the reweighted model on the (unmodified) test set.

In [10]:
X_test = test_sds.features
test_probs = model_fair.predict_proba(X_test)[:, 1]
test_pred = (test_probs > 0.5).astype(int)

## Demographic parity

We compare accuracy and demographic parity for the baseline model against the reweighed model.

In [11]:
bl_acc = accuracy(test.salary, bl_test_probs)
bl_dpd = demographic_parity_difference(
    test.salary, bl_test_pred, sensitive_features=test.sex,
)
bl_dpr = demographic_parity_ratio(
    test.salary, bl_test_pred, sensitive_features=test.sex,
)

acc = accuracy(test.salary, test_probs)
dpd = demographic_parity_difference(
    test.salary, test_pred, sensitive_features=test.sex,
)
dpr = demographic_parity_ratio(
    test.salary, test_pred, sensitive_features=test.sex,
)

print(f"Baseline accuracy: {bl_acc:.3f}")
print(f"Reweighed accuracy: {acc:.3f}\n")

print(f"Baseline demographic parity difference: {bl_dpd:.3f}")
print(f"Reweighed demographic parity difference: {dpd:.3f}\n")

print(f"Baseline demographic parity ratio: {bl_dpr:.3f}")
print(f"Reweighed demographic parity ratio: {dpr:.3f}")

Baseline accuracy: 0.853
Reweighed accuracy: 0.842

Baseline demographic parity difference: 0.193
Reweighed demographic parity difference: 0.099

Baseline demographic parity ratio: 0.290
Reweighed demographic parity ratio: 0.564


Reweighing roughly halves the demographic parity gap between men and women at a small cost to overall accuracy — a direct, visible illustration of the fairness/accuracy trade-off, without touching the model's training algorithm or its decision threshold.

In [12]:
dp_box = group_box_plots(
    np.concatenate([bl_test_probs, test_probs]),
    np.tile(test.sex.map({0: "Female", 1: "Male"}), 2),
    groups=np.concatenate(
        [np.zeros_like(bl_test_probs), np.ones_like(test_probs)]
    ),
    group_names=["Baseline", "Reweighed"],
    title="Distribution of scores by sex",
    xlabel="Scores",
    ylabel="Method",
)
dp_box

## Discussion

Reweighing is a **pre-processing** intervention: it changes what the learning algorithm sees, not the algorithm itself or its outputs. This makes it model-agnostic — any classifier that accepts sample weights can use it — and it requires no access to the protected attribute at prediction time, only at training time. Contrast this with:

- **FTU** (also pre-processing): simply removes the protected attribute as a feature. Weaker than reweighing because correlated proxy features can still leak the same information.
- **Hardt et al.** (post-processing): corrects the *decisions* of an already-trained model. Requires access to the protected attribute at prediction time, and can be shown to be optimal for equalised odds — but offers no continuous trade-off and can feel arbitrary at the level of individual decisions.

**Discussion question for students:** reweighing targets demographic parity directly by construction. Would you expect it to also improve equalised odds or calibration by sex? Try computing `equalized_odds_difference` on `test_pred` above and compare with the FTU and Hardt et al. notebooks.